# 教程 3：给每个请求挂一副自己的 LoRA

到目前为止，我们的 workload 对整台引擎只有一副面孔：探针看所有请求，训练头引导所有请求。但真实服务不是这样的——同一台引擎同时跑着成千上万个请求，你可能只想给其中一类（比如"写邮件"的）挂上一副适配器，让它的语气越服务越对味，而旁边闲聊的请求完全不受打扰。

这一篇我们就做这件事：给请求挂上各自独立的 LoRA，边服务边学。为此你会第一次碰到 PLLM 的两件新东西：

- **状态（state）**：LoRA 因子总得有个地方存、还能被更新——这就是状态，签名契约里唯一可变的东西；
- **显存预算**：状态占的是真金白银的显存，引擎不是无限大的，你要为状态申请容量，签名的计划会替你守住这条线。

教程 2 的训练头**不改模型**（只引导采样）。LoRA 不同——它直接改层的权重。这在 PLLM 里是另一种效果：**overlay**。我们从最小的一副补丁开始，一步步走到按路由共享、按判别器混合、按 chunk 调度：

1. 最小 overlay：一个低秩补丁挂在某层 MLP 上；
2. 谁和谁共享一份 LoRA：请求级状态、`keys` 与你自己的路由规则；
3. 判别器加权混合：其实只是"状态里多几个字段"；
4. prefill 内的逐步学习：wavefront 与 Jacobi 两种调度。

## 第一步：最小 overlay

先回答一个你可能已经在想的问题：LoRA 因子为什么不直接塞进请求的 KV cache？因为 KV cache 是注意力的前向缓存，由引擎按块调度，签名契约管不到它；而状态字段是契约里**唯一被承认的可变存储**——被签名、被版本化、被计入预算。想改变模型的行为，就要走 overlay 这条签了名的通道。

overlay 任务在 `Phase.IN_FORWARD` 时机运行。它拿到的输入不是 hidden states，而是**被捕获的算子三元组**：目标层的输入 `x`、输出 `y`、原始权重 `base`。下面的 `update` 做两件小事：先算出"加了低秩补丁之后"的输出（模型立刻拿它继续前向），再顺手做一步 Hebbian 风格的小更新。

两个签名约束值得现在就记住，后面每一步都靠它们：

- overlay 的状态字段用 `pllm.Buffer` 声明，形状可以依赖目标权重的尺寸（`inputs`/`outputs` 就是它的入/出维度）；
- buffer 状态**必须零初始化**——所以第一次更新之前，模型的行为和没打补丁**严格相等**。补丁带来的任何变化，都能归因于你自己写进去的更新。

还有一件隐性的好事正在发生：这副补丁占的显存，是编译期由签名计划预留出来的。默认预算取自后端观测到的可用容量；想自己收紧，可以给 `ModelConfig` 传一个 `ExecutionOptions(state_memory_bytes=...)`。那么预算超了会怎样？不会跑到一半 OOM——状态计划超出预算时，引擎在**启动阶段**就响亮地报错。这正是预算机制存在的意义：把显存事故提前成编译期的一次 `ValueError`。

跑下面的代码前，先说下预期：生成的文本会相当"破碎"。这不意外——我们的更新规则每走一步都在挪动补丁，模型被故意带偏了。真正要看的数字是最后一行的 `updates`：它等于 decode 的步数乘上目标层数（这里挂了 3 层），说明 overlay 在每一步、每一层都真的生效了。
Buffer 默认 FP32 累积，投影读取时转换到模型输入 dtype；显式 `dtype=torch.bfloat16` 可节省字段显存。


In [ ]:
import gc
import re
from typing import Annotated

import torch
import pllm
from pllm import UpdateOutput
from vllm import SamplingParams
from pllm.config import ModelConfig
from pllm.contracts.parameter_overlay import ParameterModuleKind, ParameterRole

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def config(max_num_seqs=2):
    return ModelConfig(MODEL, "bfloat16", 96, max_num_seqs, 96, 0.2, seed=7)

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

def text_of(handle, prompt, max_tokens=24, **kwargs):
    return handle.generate([prompt], SamplingParams(max_tokens=max_tokens,
        temperature=0.0, ignore_eos=True), use_tqdm=False, **kwargs)[0].outputs[0].text

LR = 0.1
RANK = 2
TARGET9 = tuple(pllm.model.ParameterTarget(layer, ParameterModuleKind.MLP,
    ParameterRole.DOWN_PROJECTION_WEIGHT) for layer in (7, 8, 9))


class MyFirstLoRA(pllm.Workload):
    """7-9 层 MLP down-projection 上的 rank-2 低秩补丁，decode 时每步更新。"""

    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    })

    @pllm.on(pllm.Phase.IN_FORWARD)
    @pllm.overlay(state="factors", targets=TARGET9,
                  operator=pllm.model.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, decode=True)
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> UpdateOutput:
        output = (x @ a.to(x.dtype).T) @ b.to(x.dtype).T                  # 投影残差，基座投影由引擎提供
        a.add_((LR * y[:, :RANK].T @ x).to(a))         # Hebbian 风格小更新
        b.add_((LR * (y.T @ y)[:, :RANK]).to(b))
        return UpdateOutput(output, changed=True, update_count=1, trained_rows=x.shape[0])


lora = pllm.make_llm(workload=MyFirstLoRA(), config=config(max_num_seqs=1))
print(text_of(lora, "The sky is"))
print("updates:", lora.diagnostics().update_count)

## 第二步：谁和谁共享 —— `keys` 与 route

刚才的 `MyFirstLoRA` 用的是 `pllm.RequestState`：每个请求一份独立状态，请求退役，状态随之退役。这回答了"不互相串扰"，但还没回答另一个问题——同一类请求，怎样共享彼此学到的东西？总不能让"写邮件"这个技能在每个新请求里都从零学起。

这正是 `keys` 的作用。默认的隔离键是请求身份；改成 `(RequestKeyField.ROUTE,)` 之后，**路由键相同的请求加入同一条 lineage**：第一个请求退役后，lineage 并不消失，而是休眠保留（它仍占着准入容量，所以 `max_num_seqs` 要留点余量）；下一个同键请求到来时直接"插队"进去，读到此前累积的全部训练。不同键之间则完全隔离，会话关闭时 lineage 才正式退役。

路由键完全由你计算，框架从不解释它——正则、租户 ID、判别器分数都行。我们在 `generate(..., routes=(...))` 里传入；下面用一个邮箱正则：带邮箱的 prompt 归 `"email"`，其余归 `"plain"`。顺带一提：忘了传路由值的请求会在准入时被拒，而不是悄悄混进别人的 lineage。

顺便把两种作用域分清，之后不会再混：`pllm.InstanceState` 是全引擎一条 lineage，所有请求共享（教程 1 里那种只读编码器就是这个气质）；`pllm.RequestState` 是每个准入键一条 lineage。LoRA 要按请求分家，自然选后者。

跑之前先想好三个可检验的断言，代码会逐一验证：

- 同一 route 第二次生成 ≠ 第一次（它读到了自己累积的训练）；
- 另一个 route 的第二次生成也 ≠ 它的第一次（它自己也在积累，不是只有 email 在学）；
- 全新 route 的第一次生成 == 任何训练发生之前的基线（隔离是严格的，别人的训练漏不进来）。

In [ ]:
class RoutedLoRA(MyFirstLoRA):
    # 唯一的区别：lineage 按 route 键组织，跨请求共享
    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    }, keys=(pllm.state.RequestKeyField.ROUTE,))


EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w]+")

def route_of(prompt: str) -> str:
    return "email" if EMAIL.search(prompt) else "plain"


lora.close()
cleanup()
lora = pllm.make_llm(workload=RoutedLoRA(), config=config(max_num_seqs=4))
email_prompt = "Write a reply to alice@example.com about the meeting"
plain_prompt = "The sky is"

plain_1 = text_of(lora, plain_prompt, routes=("plain",))
email_1 = text_of(lora, email_prompt, routes=(route_of(email_prompt),))
plain_2 = text_of(lora, plain_prompt, routes=("plain",))
email_2 = text_of(lora, email_prompt, routes=(route_of(email_prompt),))
fresh = text_of(lora, plain_prompt, routes=("another",))

print("email 第 1 次:", email_1)
print("email 第 2 次:", email_2)
assert email_1 != email_2        # 同 route：读到累积的训练
assert plain_1 != plain_2        # plain 自己也积累了
assert fresh == plain_1          # 新 route 从零开始，不受他人训练影响
print("route 共享与隔离都得证")

## 第三步：判别器加权混合 —— 只是"状态里多几个字段"

两路 LoRA 不该非此即彼的时候，直觉的做法是：让一个小判别器 MLP 看一眼当前 hidden，给两路各打一个分，softmax 加权混合。

在 PLLM 里，这**不需要任何新机制**：一个状态声明里放两路因子加判别器权重，`forward` 里写混合数学，训练骨架直接沿用教程 2 的（`BEFORE_SAMPLE` 预测引导 + `AFTER_SAMPLE` 训练）。在状态系统眼里，"一路 LoRA"和"两路 LoRA 加判别器"只是字段列表长短的区别。

唯一要留神的是参数依然**按行就位**：`pllm.Parameter` 读出来的张量带着领先的行轴——每一行拿到的是自己 lineage 的版本——所以 `a1` 实际是 `(行数, H, RANK)`，矩阵乘法一律写成 `bmm` 的按行形式，别用裸 `matmul`。

另外注意 `ready` 这道小门：`StateVersion` 给出每个 lineage 的版本号，`versions > 0` 即"这条 lineage 至少训练过一次"。没训练过的行不参与引导——否则随机初始化的混合头会注入毫无意义的修正。

输出里有个数字值得多看一眼：`fallback: 0`。它断言整段运行没有任何一次执行溜出签名原语之外。混合、多头、判别器……花样再多，纪律不变。

In [3]:
class MixtureLoRA(pllm.Workload):
    """判别器对两路 LoRA 头的输出做 softmax 加权；所有字段一起训练。"""

    mixture = pllm.RequestState(
        {"a1": lambda m: (m.hidden_size, RANK), "b1": lambda m: (RANK, m.hidden_size),
         "a2": lambda m: (m.hidden_size, RANK), "b2": lambda m: (RANK, m.hidden_size),
         "disc_w": lambda m: (m.hidden_size, 2), "disc_b": (2,)},
        dtype=torch.float32,
        initialize=lambda tensors, generator: [
            torch.nn.init.normal_(tensor, std=0.02, generator=generator)
            for tensor in tensors.values()],
        optimizer=pllm.AdamW(lr=3e-3),
        keys=(pllm.state.RequestKeyField.ROUTE,))

    def forward(self, h, a1, b1, a2, b2, disc_w, disc_b):
        x = h.float().unsqueeze(1)                                            # (rows, 1, H)
        weights = torch.softmax(torch.bmm(x, disc_w).squeeze(1) + disc_b, -1) # (rows, 2)
        out1 = torch.bmm(torch.bmm(x, a1), b1).squeeze(1)
        out2 = torch.bmm(torch.bmm(x, a2), b2).squeeze(1)
        return weights[:, 0:1] * out1 + weights[:, 1:2] * out2

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("mixture")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, h: Annotated[torch.Tensor, pllm.Hidden(-1)],
                a1: Annotated[torch.Tensor, pllm.Parameter("mixture", "a1")],
                b1: Annotated[torch.Tensor, pllm.Parameter("mixture", "b1")],
                a2: Annotated[torch.Tensor, pllm.Parameter("mixture", "a2")],
                b2: Annotated[torch.Tensor, pllm.Parameter("mixture", "b2")],
                disc_w: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_w")],
                disc_b: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_b")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("mixture")]) -> torch.Tensor:
        return self.forward(h, a1, b1, a2, b2, disc_w, disc_b)

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="mixture", reduction="state_mean")
    def learn(self, h: Annotated[torch.Tensor, pllm.Hidden(-1)],
              target: Annotated[torch.Tensor, pllm.Hidden(0)],
              a1: Annotated[torch.Tensor, pllm.Parameter("mixture", "a1")],
              b1: Annotated[torch.Tensor, pllm.Parameter("mixture", "b1")],
              a2: Annotated[torch.Tensor, pllm.Parameter("mixture", "a2")],
              b2: Annotated[torch.Tensor, pllm.Parameter("mixture", "b2")],
              disc_w: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_w")],
              disc_b: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_b")]) -> torch.Tensor:
        losses = (self.forward(h, a1, b1, a2, b2, disc_w, disc_b) - target.float()).square().mean(1)
        return losses  # 每行一个损失，pllm 按状态求均值


lora.close()
cleanup()
mixture = pllm.make_llm(workload=MixtureLoRA(), config=config(max_num_seqs=4))
print("email:", text_of(mixture, email_prompt, routes=("email",)))
print("plain:", text_of(mixture, plain_prompt, routes=("plain",)))
d = mixture.diagnostics()
print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
mixture.close()

INFO 09-10 18:42:17 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 96, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 96, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:42:17 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:42:18 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:42:18 [model.py:1883] Using max model len 96


INFO 09-10 18:42:18 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 18:42:19 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=96, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (8 outputs left out)

INFO 09-10 18:42:22 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.89 s


WARNING 09-10 18:42:22 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


email: . Alice is a software engineer who works at a tech company. She is currently working on a project that involves developing a


plain:  a beautiful place. It is a place of endless possibilities. The sky is a place of endless possibilities for people to explore
losses: 46 updates: 46 fallback: 0


## 第四步：prefill 内的逐步学习 —— wavefront 与 Jacobi

到目前为止，学习都踩在 decode 的步进节奏上：一步一个 token，一步一次更新。但 prefill 一次要处理一整批 token。把这批按 chunk 切开后，一个经典问题浮出水面：**后面 chunk 的前向，要不要立刻看到前面 chunk 刚算出的更新？** 两种经典调度给出相反的答案：

- **wavefront（边算边改）**：chunk k+1 的前向立即看到 chunk k 的更新，像流水线一样逐步复合；
- **Jacobi（对齐再改）**：所有 chunk 都对同一份"输入版本"的因子求值，更新攒到最后一次性施加。

在 PLLM 里，这两种调度**只是 `update` 循环写法的差别**——发布纪律、状态语义完全不变，你不用为其中任何一种学新 API。为了让差别可观测，我们故意把更新规则写成不对称的：`a` 的增量与当前因子无关（第一个 chunk 就会移动它），而 `b` 的增量依赖**当前的 `a`**。于是 wavefront 里 `b` 看到的是逐步更新后的 `a`，一路复合；Jacobi 里 `b` 看到的始终是进门时的那份，停在输入版本。

预期输出：两种调度的 update 计数完全相同（同样的 chunk 数、同样的层数），但生成的文本不一样——同样的数据、同样的学习率，仅仅是"什么时候 `add_`"的差别，就足以把模型带向两个方向。

In [ ]:
LAYERS = (1, 3, 5, 7, 9)
TARGETS = tuple(pllm.model.ParameterTarget(layer, ParameterModuleKind.MLP,
    ParameterRole.DOWN_PROJECTION_WEIGHT) for layer in LAYERS)
CHUNK = 8


class Wavefront(pllm.Workload):
    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    })

    @pllm.on(pllm.Phase.IN_FORWARD)
    @pllm.overlay(state="factors", targets=TARGETS,
                  operator=pllm.model.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, prefill=True, decode=False)
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> UpdateOutput:
        outputs = []
        for start in range(0, x.shape[0], CHUNK):
            xc, yc = x[start:start + CHUNK], y[start:start + CHUNK]
            outputs.append((xc @ a.to(xc.dtype).T) @ b.to(xc.dtype).T)         # 当前因子立即生效
            a.add_((LR * yc[:, :RANK].T @ xc).to(a))
            b.add_((LR * yc.T @ (xc @ a.to(xc.dtype).T)).to(b))        # 看到刚更新的 a
        return UpdateOutput(torch.cat(outputs), changed=True,
                            update_count=len(outputs), trained_rows=x.shape[0])


class Jacobi(pllm.Workload):
    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    })

    @pllm.on(pllm.Phase.IN_FORWARD)
    @pllm.overlay(state="factors", targets=TARGETS,
                  operator=pllm.model.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, prefill=True, decode=False)
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> UpdateOutput:
        outputs = [(x[start:start + CHUNK] @ a.to(x.dtype).T) @ b.to(x.dtype).T
                   for start in range(0, x.shape[0], CHUNK)]  # 全部对输入版本求值
        grad_a = torch.zeros_like(a)
        grad_b = torch.zeros_like(b)
        for start in range(0, x.shape[0], CHUNK):
            xc, yc = x[start:start + CHUNK], y[start:start + CHUNK]
            grad_a += yc[:, :RANK].T @ xc
            grad_b += yc.T @ (xc @ a.to(xc.dtype).T)                   # a 还是输入版本
        a.add_((LR * grad_a).to(a))
        b.add_((LR * grad_b).to(b))
        return UpdateOutput(torch.cat(outputs), changed=True,
                            update_count=len(outputs), trained_rows=x.shape[0])


cleanup()
long_prompt = "The quick brown fox jumps over the lazy dog near the river bank"
wave = pllm.make_llm(workload=Wavefront(), config=config(max_num_seqs=1))
text_wave = text_of(wave, long_prompt)
updates_wave = wave.diagnostics().update_count
wave.close()
cleanup()
jac = pllm.make_llm(workload=Jacobi(), config=config(max_num_seqs=1))
text_jac = text_of(jac, long_prompt)
updates_jac = jac.diagnostics().update_count
jac.close()

print("wavefront:", text_wave, "updates:", updates_wave)
print("jacobi:   ", text_jac, "updates:", updates_jac)
assert text_wave != text_jac and updates_wave == updates_jac > 0

## 小结

这一篇信息量不小，值得带走的是这几条：

- overlay 任务读的是**算子三元组**（输入 / 输出 / 原权重），它的输出直接进模型前向——这是签名契约里"改模型"的正规通道；
- buffer 状态零初始化是签名约束：初始行为严格等于无补丁，之后的一切变化都可归因于你写的更新；
- **instance state 全引擎一条 lineage，request state 每个准入键一条**；`keys` 决定谁和谁共享，route 键由你计算，框架只看字符串；
- 状态占用真实显存，容量在编译期由签名计划预留；预算（`state_memory_bytes`）超了会在引擎启动时响亮报错，而不是运行到一半翻车；
- 混合、多头、判别器……都是"状态里多几个字段 + `forward` 里多几行数学"；参数按行就位，矩阵运算记得用 `bmm`；
- wavefront / Jacobi 的差别只在 `update` 循环里"什么时候 `add_`"，发布纪律不变。

到现在，单个请求（或一组同 route 的请求）的一生已经被状态串起来了。但如果一段交互要**来回很多轮**、每一轮都接着上一轮学到的继续呢？下一篇我们看 overlay 与 replay 支撑的多轮循环：[教程 4：三种循环](04_loops.ipynb)。